<a href="https://colab.research.google.com/github/anjalikrishnaak6-research/neuroimag_EEG/blob/main/EEG_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install mne-bids

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 10.3 MB/s eta 0:00:00


In [43]:
import os
import numpy as np
import mne
from mne_bids import BIDSPath, read_raw_bids
from google.colab import drive


In [ ]:
from mne_bids import BIDSPath, read_raw_bids
import os

root = "/content/drive/MyDrive/VEP_dataset"

In [ ]:
subjects = [f"{i:03d}" for i in range(1, 7)]
sessions = ["01", "02"]
TASK_TO_ANALYZE = "SFSP"

all_X = []
all_y = []
all_classical = []

In [ ]:
for sub in subjects:
    for ses in sessions:

        print(f"\nLoading sub-{sub} ses-{ses}")

        bids_path = BIDSPath(
            subject=sub,
            session=ses,
            task=TASK_TO_ANALYZE,
            datatype='eeg',
            root=root
        )

        try:
            raw = read_raw_bids(bids_path=bids_path, preload=True)
        except Exception as e:
            print("Skipping:", e)
            continue

        # BASIC PREPROCESSING

        raw.filter(0.1, 30., fir_design='firwin')
        raw.notch_filter(freqs=50)
        raw.set_eeg_reference('average', projection=False)

        # ICA (NO ARBITRARY EXCLUSION)

        ica = mne.preprocessing.ICA(
            n_components=20,
            random_state=97,
            max_iter='auto'
        )

        ica.fit(raw)

        # OPTIONAL: Uncomment for manual inspection
        # ica.plot_components()
        # ica.plot_sources(raw)

        # For automation: apply without excluding components
        raw_clean = ica.apply(raw.copy())

        # EVENTS

        events, event_id = mne.events_from_annotations(raw_clean)

        print("Detected events:", event_id)

        # Automatically infer first two conditions
        if len(event_id) < 2:
            print("Not enough conditions found.")
            continue

        condition_names = list(event_id.keys())
        cond1 = condition_names[0]
        cond2 = condition_names[1]

        event_dict = {
            cond1: event_id[cond1],
            cond2: event_id[cond2]
        }

        print("Using conditions:", event_dict)

        # EPOCHING

        epochs = mne.Epochs(
            raw_clean,
            events,
            event_id=event_dict,
            tmin=-0.2,
            tmax=0.8,
            baseline=(-0.2, 0),
            preload=True
        )

        if len(epochs) == 0:
            continue

        # DEEP LEARNING DATA

        X = epochs.get_data()
        y_raw = epochs.events[:, -1]

        label_map = {
            event_dict[cond1]: 0,
            event_dict[cond2]: 1
        }

        y = np.array([label_map[val] for val in y_raw])

        all_X.append(X)
        all_y.append(y)

        # CLASSICAL P3 FEATURES

        times = epochs.times
        p3_window = np.where((times >= 0.3) & (times <= 0.6))[0]

        roi_channels = ['Cz', 'CPz', 'Pz', 'CP1', 'CP2', 'P3', 'P4']
        roi_indices = [
            epochs.ch_names.index(ch)
            for ch in roi_channels
            if ch in epochs.ch_names
        ]

        if len(roi_indices) == 0:
            continue

        X_data = X

        p3_mean_amp = X_data[:, roi_indices][:, :, p3_window].mean(axis=(1,2))
        p3_peak_amp = X_data[:, roi_indices][:, :, p3_window].max(axis=(1,2))

        p3_peak_latency = []
        for trial in X_data[:, roi_indices][:, :, p3_window]:
            mean_roi = trial.mean(axis=0)
            peak_idx = np.argmax(mean_roi)
            latency = times[p3_window][peak_idx]
            p3_peak_latency.append(latency)

        p3_peak_latency = np.array(p3_peak_latency)

        X_classical = np.column_stack([
            p3_mean_amp,
            p3_peak_amp,
            p3_peak_latency
        ])

        all_classical.append(X_classical)

In [ ]:
if len(all_X) == 0:
    raise RuntimeError("No valid epochs were extracted.")

X_final = np.concatenate(all_X, axis=0)
y_final = np.concatenate(all_y, axis=0)
X_classical_final = np.concatenate(all_classical, axis=0)

print("Final DL tensor:", X_final.shape)
print("Final labels:", y_final.shape)
print("Final classical:", X_classical_final.shape)


In [ ]:
np.save("/content/drive/MyDrive/X_eeg_all.npy", X_final)
np.save("/content/drive/MyDrive/y_all.npy", y_final)
np.save("/content/drive/MyDrive/X_classical_all.npy", X_classical_final)

print("Saved successfully.")